<a href="https://colab.research.google.com/github/eliabrodsky/la_data/blob/main/LA_Rural_Hospital_EDA_Charts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Louisiana Rural Hospitals: Exploratory Analysis

24 charts across 64 rural hospitals, built from raw source files.

| Source | What it provides |
|---|---|
| CMS Hospital Provider Cost Report, FY2021&ndash;FY2023 | Balance sheet, income statement, utilization, payer day mix |
| LDH Rural Transformation payment file, SFY2024&ndash;SFY2025 | Directed payments, cost settlement, UPL, DSH, average daily census |
| Louisiana Health Atlas | ZIP population, rurality, social vulnerability, facility counts |
| LDH report 109A | Medicaid managed care enrollment by parish |
| LIHNC roster and Epic state instance pipeline | Program membership and implementation economics |

**Three normalizations run throughout.** Size, so a 15-bed CAH is comparable to a 129-bed referral
center. Ownership, because balance-sheet measures reflect treasury structure. Reporting period,
because six cost reports cover under 330 days and one covers 32.

## Setup

In [1]:
# ============ SETUP + HELPERS (tested standalone) ============
import numpy as np, pandas as pd, plotly.graph_objects as go, plotly.express as px
from plotly.subplots import make_subplots

NAVY,TEAL,GOLD,RED,GREEN,BLUE,GREY = '#1F3864','#1C7293','#B08511','#E24B4A','#1D9E75','#378ADD','#888780'
OWNER = {'Government':NAVY,'Nonprofit':TEAL,'Proprietary':GOLD}
PROGRAM = {'Both':NAVY,'LIHNC only':TEAL,'Epic only':GOLD,'Neither':GREY}
RURAL = {'Metro / adjacent':NAVY,'Micropolitan':TEAL,'Small town / isolated':GOLD}

def style(fig, title, subtitle='', legend=None, height=520, left=90, bottom=None):
    """Consistent styling. Legend sits BELOW the plot so it can never overlap
    the chart area or the title. bottom margin grows to make room for it."""
    show_legend = legend is not False and any(
        tr.showlegend is not False and getattr(tr, 'name', None) for tr in fig.data)
    bottom = bottom if bottom is not None else (150 if show_legend else 80)
    fig.update_layout(
        title=dict(text=f'<b>{title}</b>' + (f'<br><sub>{subtitle}</sub>' if subtitle else ''),
                   x=0, xanchor='left', y=0.97, yanchor='top'),
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(size=12, family='Aptos, Arial'),
        height=height, margin=dict(l=left, r=45, t=105, b=bottom),
        legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='left', x=0,
                    title='', bgcolor='rgba(0,0,0,0)') if show_legend else dict(),
        showlegend=show_legend)
    fig.update_xaxes(gridcolor='#e8e8e2', zeroline=False)
    fig.update_yaxes(gridcolor='#e8e8e2', zeroline=False)
    return fig

def quadrants(fig, x, y, labels, xr, yr):
    """Draw quadrant dividers and label each corner.
    labels order: bottom-left, bottom-right, top-left, top-right."""
    fig.add_vline(x=x, line=dict(color=GREY, width=1, dash='dash'))
    fig.add_hline(y=y, line=dict(color=GREY, width=1, dash='dash'))
    pos = [(xr[0], yr[0], 'left', 'bottom'), (xr[1], yr[0], 'right', 'bottom'),
           (xr[0], yr[1], 'left', 'top'),    (xr[1], yr[1], 'right', 'top')]
    for txt, (px_, py_, ax_, ay_) in zip(labels, pos):
        if not txt:
            continue
        fig.add_annotation(x=px_, y=py_, text=f'<b>{txt}</b>', showarrow=False,
                           xanchor=ax_, yanchor=ay_, font=dict(size=10, color=GREY),
                           bgcolor='rgba(255,255,255,0.75)')
    return fig

## Load the raw sources

Read directly from GitHub so the notebook runs from a clean Colab session.

In [2]:
# ============ LOAD FROM RAW ============
BASE = 'https://raw.githubusercontent.com/eliabrodsky/la_data/main/'
FILES = {2021:'CostReport_2021_Final.csv',2022:'CostReport_2022_Final.csv',2023:'CostReport_2023_Final.csv'}
CONTROL = {1:'Nonprofit-Church',2:'Nonprofit-Other',3:'Proprietary-Individual',4:'Proprietary-Corp',
           5:'Proprietary-Partnership',6:'Proprietary-Other',7:'Gov-Federal',8:'Gov-City-County',
           9:'Gov-County',10:'Gov-State',11:'Gov-Hospital District',12:'Gov-City',13:'Gov-Other'}
PO_BOX = {'70511':'70510','70562':'70560','71121':'71220','70157':'70517','70308':'70380'}

def load_cost_report(year, path):
    d = pd.read_table(BASE+path, sep=',', header=0, low_memory=False)
    d = d[d['State Code']=='LA'].copy()
    col = lambda n: pd.to_numeric(d[n],errors='coerce') if n in d.columns else pd.Series(np.nan,index=d.index)
    begin=pd.to_datetime(d['Fiscal Year Begin Date'],errors='coerce')
    end=pd.to_datetime(d['Fiscal Year End Date'],errors='coerce')
    period=(end-begin).dt.days.clip(lower=1)
    cash=col('Cash on Hand and in Banks').fillna(0)+col('Temporary Investments').fillna(0)
    opex,dep=col('Less Total Operating Expense'),col('Depreciation Cost').fillna(0)
    npr,oth=col('Net Patient Revenue'),col('Total Other Income').fillna(0)
    total_days=col('Total Days (V + XVIII + XIX + Unknown)')
    o=pd.DataFrame({'fy':year,
        'ccn':d['Provider CCN'].astype(str).str.zfill(6),
        'hospital_filing_name':d['Hospital Name'].str.strip(),
        'city':d['City'].str.strip().str.title(),
        'medicare_class':d['CCN Facility Type'],
        'control':pd.to_numeric(d['Type of Control'],errors='coerce').map(CONTROL),
        'zip':d['Zip Code'].astype(str).str.extract(r'(\d{5})')[0].replace(PO_BOX),
        'reporting_days':period,'beds':col('Number of Beds'),'fte':col('FTE - Employees on Payroll'),
        'net_patient_revenue':npr,'operating_expense':opex,
        'medicaid_revenue':col('Net Revenue from Medicaid'),
        'uncompensated_care_cost':col('Cost of Uncompensated Care'),
        'cost_to_charge':col('Cost To Charge Ratio')})
    o['days_cash_on_hand']=(cash/((opex-dep)/period)).where((opex-dep)>0)
    o['equity_ratio']=(col('Total Fund Balances')/col('Total Assets')).where(col('Total Assets')>0)
    o['current_ratio']=(col('Total Current Assets')/col('Total Current Liabilities')).where(col('Total Current Liabilities')>0)
    o['patient_services_margin']=(col('Net Income from Service to Patients')/npr).where(npr>0)
    o['total_margin']=(col('Net Income')/(npr+oth)).where((npr+oth)>0)
    o['medicaid_days_share']=(col('Total Days Title XIX')/total_days).where(total_days>0)
    o['medicare_days_share']=(col('Total Days Title XVIII')/total_days).where(total_days>0)
    print(f'  FY{year}: {len(o)} Louisiana hospitals')
    return o

print('Loading CMS cost reports')
panel = pd.concat([load_cost_report(y,p) for y,p in FILES.items()], ignore_index=True)

xwalk = pd.read_table(BASE+'rural_ccn.csv', sep=',', header=0, dtype=str)
xwalk['ccn']=xwalk['ccn'].str.zfill(6)
panel = panel[panel.ccn.isin(set(xwalk.ccn))].copy()
panel['ownership'] = panel.control.map(lambda c:'Government' if str(c).startswith('Gov')
    else ('Proprietary' if str(c).startswith('Proprietary') else 'Nonprofit'))
print(f'{len(panel)} hospital-years across {panel.ccn.nunique()} rural hospitals')

pay = pd.read_table(BASE+'ldh_payments_2024_2025.csv',sep=',',header=0,low_memory=False)
pay['ccn']=pay.ccn.astype(str).str.zfill(6)
for y in (2024,2025):
    pay[f'ldh_payments_{y}']=pay[[f'dp_{y}',f'cs_{y}',f'upl_{y}',f'dsh_{y}']].fillna(0).sum(axis=1)

atlas = pd.read_table(BASE+'louisiana_health_atlas_export_zip.csv',sep=',',header=0,low_memory=False)
atlas['zip']=atlas['ZIP Code'].astype(str).str.zfill(5)
atlas=atlas.drop_duplicates('zip').rename(columns={
    'Population':'zip_population','Classification':'zip_classification',
    'Rural Designation (RUCA Category)':'ruca','Broadband Deserts':'broadband_desert_pct',
    'Social Vulnerability (Poverty)':'poverty_pct','Social Vulnerability (Food Access)':'food_access_pct',
    'Transportation (Vehicle Ownership)':'no_vehicle_pct',
    'Healthcare Facility (Specialty)':'specialty_facilities_zip',
    'Healthcare Facility (Acute)':'acute_facilities_zip','Diabetes Prevalence':'diabetes_prevalence'})
mco = pd.read_table(BASE+'parish_mco_enrollment.csv',sep=',',header=0)
print(f'{len(pay)} hospitals with LDH payments | {len(atlas)} ZIPs | {len(mco)} parishes')

Loading CMS cost reports
  FY2021: 209 Louisiana hospitals
  FY2022: 209 Louisiana hospitals
  FY2023: 205 Louisiana hospitals
191 hospital-years across 64 rural hospitals
58 hospitals with LDH payments | 518 ZIPs | 64 parishes


## Build the analysis table

One row per hospital: three years of financial measures widened, joined to payments, place and program membership, then normalized.

In [3]:
# ============ BUILD THE ANALYSIS TABLE ============
latest = panel.sort_values('fy').groupby('ccn').tail(1).set_index('ccn')
wide = panel.pivot_table(index='ccn',columns='fy',
    values=['days_cash_on_hand','equity_ratio','current_ratio','patient_services_margin',
            'total_margin','net_patient_revenue'],aggfunc='first')
wide.columns=[f'{a}_fy{b}' for a,b in wide.columns]

XW = ['ccn','hospital','parish','lihnc_member','in_epic_pipeline','epic_wave','epic_status',
      'epic_est_pricing_m','epic_amb_vol','epic_ip_vol','epic_providers','medicaid_class']
XW = [c for c in XW if c in xwalk.columns]

h = xwalk[XW].set_index('ccn').join(wide).join(latest[[
    'city','zip','medicare_class','control','ownership','beds','fte','reporting_days',
    'net_patient_revenue','medicaid_revenue','uncompensated_care_cost','cost_to_charge',
    'medicaid_days_share','medicare_days_share']]).reset_index()

for c in ['epic_est_pricing_m','epic_amb_vol','epic_ip_vol','epic_providers']:
    h[c]=pd.to_numeric(h[c],errors='coerce')

h = h.merge(pay[['ccn','dp_2024','dp_2025','cs_2025','upl_2025','dsh_2025',
                 'ldh_payments_2024','ldh_payments_2025','adc_2024','discharges_2024']],
            on='ccn',how='left')
h = h.merge(atlas[['zip','zip_population','zip_classification','ruca','broadband_desert_pct',
                   'poverty_pct','food_access_pct','no_vehicle_pct','specialty_facilities_zip',
                   'acute_facilities_zip','diabetes_prevalence']],on='zip',how='left')
h['parish']=h.parish.astype(str).str.upper().str.replace('.','',regex=False)
h = h.merge(mco,on='parish',how='left')

# ---- normalization ----
h['medicare_class']=h.medicare_class.fillna('Unclassified')
h['program']=np.select(
    [(h.lihnc_member=='Yes')&(h.in_epic_pipeline=='Yes'),(h.lihnc_member=='Yes'),(h.in_epic_pipeline=='Yes')],
    ['Both','LIHNC only','Epic only'],default='Neither')
h['rurality_band']=pd.cut(h.ruca,[0,3,6,10],labels=['Metro / adjacent','Micropolitan','Small town / isolated'])

# size-normalized
h['revenue_per_bed']=h.net_patient_revenue_fy2023/h.beds.replace(0,np.nan)
h['revenue_per_fte']=h.net_patient_revenue_fy2023/h.fte.replace(0,np.nan)
h['fte_per_bed']=h.fte/h.beds.replace(0,np.nan)
h['occupancy_rate']=h.adc_2024/h.beds.replace(0,np.nan)

# share-of-revenue measures
h['medicaid_share_of_revenue']=h.medicaid_revenue/h.net_patient_revenue_fy2023
h['uncompensated_share_of_revenue']=h.uncompensated_care_cost/h.net_patient_revenue_fy2023
h['state_payment_share_of_revenue']=h.ldh_payments_2025/h.net_patient_revenue_fy2023
h['epic_cost_share_of_revenue']=h.epic_est_pricing_m*1e6/h.net_patient_revenue_fy2023
h['epic_cost_per_provider']=h.epic_est_pricing_m*1e6/h.epic_providers

# derived
h['non_care_margin_gap']=h.total_margin_fy2023-h.patient_services_margin_fy2023
h['days_cash_change_21_23']=h.days_cash_on_hand_fy2023-h.days_cash_on_hand_fy2021
h['medicaid_revenue_at_risk']=h.medicaid_revenue*h.mco_pct_change_24_26.abs()
h['social_vulnerability_index']=h[['poverty_pct','food_access_pct','no_vehicle_pct','broadband_desert_pct']].mean(axis=1)

# within-ownership percentile ranks
for c in ['days_cash_on_hand_fy2023','equity_ratio_fy2023','current_ratio_fy2023']:
    h[c+'_rank']=h.groupby('ownership')[c].rank(pct=True)

h=h.copy()
print(f'analysis table: {h.shape[0]} hospitals, {h.shape[1]} columns')
print(h.program.value_counts().to_string())

analysis table: 64 hospitals, 86 columns
program
Neither       31
Both          14
LIHNC only    10
Epic only      9


## Collect the figures

Each chart appends to `FIGS` so the whole set can be exported at the end.

In [4]:
FIGS = []

def add(fig, note):
    """Store a figure with its reading note."""
    FIGS.append((fig, note))
    fig.show()
    print(note)
    print()

---
# Part 1 &nbsp; What the numbers measure before they measure hospitals

Three charts establish what is being measured. The second is the one that constrains everything after it: liquidity differs by ownership structure, not by hospital performance, so raw days cash cannot be compared across ownership types.

In [5]:
# 1
c=h.groupby(['medicare_class','ownership']).size().reset_index(name='hospitals')
f=px.bar(c,x='medicare_class',y='hospitals',color='ownership',color_discrete_map=OWNER,barmode='stack')
f.update_xaxes(title='Medicare reimbursement class'); f.update_yaxes(title='Hospitals')
add(style(f,'1. The universe','64 rural hospitals by Medicare reimbursement class and ownership type'),
 'Government hospital districts dominate. Proprietary hospitals concentrate in the IPPS class and are nearly absent from CAH.')

Government hospital districts dominate. Proprietary hospitals concentrate in the IPPS class and are nearly absent from CAH.



In [6]:
# 2
f=go.Figure()
for s_,c_ in OWNER.items():
    f.add_trace(go.Box(y=h.loc[h.ownership==s_,'days_cash_on_hand_fy2023'],name=s_,marker_color=c_,
                       boxpoints='all',jitter=.4,pointpos=0,showlegend=False))
f.update_yaxes(title='Days cash on hand (FY2023)',range=[-200,450]); f.update_xaxes(title='Ownership type')
add(style(f,'2. Liquidity measures ownership structure, not solvency',
   'Days cash on hand = (cash + temporary investments) / daily cash operating expense',legend=False),
 'Median 110 days for government districts against 4.5 for proprietary. Kruskal-Wallis p=0.0001 by ownership, p=0.22 by Medicare class. Districts hold their own cash; corporate hospitals sweep it to a parent. Every balance-sheet comparison below is ranked within ownership for this reason.')

Median 110 days for government districts against 4.5 for proprietary. Kruskal-Wallis p=0.0001 by ownership, p=0.22 by Medicare class. Districts hold their own cash; corporate hospitals sweep it to a parent. Every balance-sheet comparison below is ranked within ownership for this reason.



In [7]:
# 3
m=h.melt(id_vars='medicare_class',value_vars=['patient_services_margin_fy2023','total_margin_fy2023'],
         var_name='measure',value_name='margin')
m['measure']=m.measure.map({'patient_services_margin_fy2023':'Patient services margin',
                            'total_margin_fy2023':'Total margin'})
f=px.box(m[m.margin.between(-1,.6)],x='medicare_class',y='margin',color='measure',
         color_discrete_map={'Patient services margin':RED,'Total margin':GREEN})
f.update_yaxes(title='Margin',tickformat='.0%'); f.update_xaxes(title='Medicare reimbursement class')
add(style(f,'3. Every class loses money delivering care',
   'Patient services margin is net income from patient care over net patient revenue. Total margin adds all other income'),
 'The red boxes sit below zero for every class. The green boxes sit above for CAH and SCH. What separates them is revenue that does not come from delivering care.')

The red boxes sit below zero for every class. The green boxes sit above for CAH and SCH. What separates them is revenue that does not come from delivering care.



---
# Part 2 &nbsp; Where the money comes from

Every class loses money delivering care. These charts identify what fills the gap.

In [8]:
# 4
x=h.dropna(subset=['state_payment_share_of_revenue','non_care_margin_gap'])
x=x[x.non_care_margin_gap.between(-.2,1)]
f=px.scatter(x,x='state_payment_share_of_revenue',y='non_care_margin_gap',color='ownership',
    size='net_patient_revenue_fy2023',hover_name='hospital',color_discrete_map=OWNER,
    trendline='ols',trendline_scope='overall',trendline_color_override=GREY,size_max=30)
f.update_xaxes(title='State Medicaid payments as a share of net patient revenue',tickformat='.0%')
f.update_yaxes(title='Non-care margin gap',tickformat='.0%')
add(style(f,'4. State payments explain most of the non-care margin gap',
   'Non-care margin gap = total margin minus patient services margin, the part not earned from delivering care'),
 'rho=+0.48, p=0.0003. Median payments 13.9% of revenue against a median gap of 18.4%, so roughly three-quarters of the gap is state money. Government districts sit above the line because they also collect ad valorem tax.')

rho=+0.48, p=0.0003. Median payments 13.9% of revenue against a median gap of 18.4%, so roughly three-quarters of the gap is state money. Government districts sit above the line because they also collect ad valorem tax.



In [9]:
# 5
comp=pd.DataFrame({'stream':['Directed payments','DSH','UPL','Cost settlement'],
    'amount':[h.dp_2025.sum(),h.dsh_2025.sum(),h.upl_2025.sum(),h.cs_2025.sum()]})
f=px.bar(comp,x='amount',y='stream',orientation='h',color_discrete_sequence=[NAVY])
f.update_traces(text=comp.amount.apply(lambda v:f'${v/1e6:.1f}M'),textposition='outside')
f.update_xaxes(title='SFY2025 payments to these hospitals',tickprefix='$',range=[0,3.2e8])
f.update_yaxes(title='Payment stream')
add(style(f,'5. One program carries the state money','SFY2025 Medicaid payments by stream',left=150,height=420),
 'Directed payments are 95% of the total. Cost settlement, usually credited with keeping cost-based hospitals solvent, is under $1M. The SFY2029 exposure is one program renewed through MCO contracts.')

Directed payments are 95% of the total. Cost settlement, usually credited with keeping cost-based hospitals solvent, is under $1M. The SFY2029 exposure is one program renewed through MCO contracts.



---
# Part 3 &nbsp; Direction of travel

Three years of the same hospitals. Aggregate trend and per-hospital trend disagree, and both are reported.

In [10]:
# 6
tr=pd.DataFrame([{'fy':y,'Days cash on hand':h[f'days_cash_on_hand_fy{y}'].median(),
    'Patient services margin':h[f'patient_services_margin_fy{y}'].median()*100,
    'Total margin':h[f'total_margin_fy{y}'].median()*100} for y in (2021,2022,2023)])
f=make_subplots(specs=[[{'secondary_y':True}]])
f.add_trace(go.Scatter(x=tr.fy,y=tr['Days cash on hand'],name='Days cash on hand',
    line=dict(color=NAVY,width=3),mode='lines+markers'))
for n,c_ in [('Patient services margin',RED),('Total margin',GREEN)]:
    f.add_trace(go.Scatter(x=tr.fy,y=tr[n],name=n+' (%)',line=dict(color=c_,width=2,dash='dot'),
        mode='lines+markers'),secondary_y=True)
f.update_xaxes(title='Fiscal year',tickvals=[2021,2022,2023])
f.update_yaxes(title='Days cash on hand',secondary_y=False)
f.update_yaxes(title='Margin (%)',secondary_y=True,gridcolor='rgba(0,0,0,0)')
add(style(f,'6. Reserves are falling, margins are not','Medians across the same 64 hospitals, FY2021 to FY2023'),
 'Median days cash fell from 103.5 to 57.3 while margins barely moved. The decline is concentrated in the top half of the distribution.')

Median days cash fell from 103.5 to 57.3 while margins barely moved. The decline is concentrated in the top half of the distribution.



In [11]:
# 7
w=h.dropna(subset=['days_cash_on_hand_fy2021','days_cash_change_21_23']).copy()
w['quartile']=pd.qcut(w.days_cash_on_hand_fy2021,4,labels=['Q1 lowest','Q2','Q3','Q4 highest'])
b=w.groupby('quartile',observed=True)['days_cash_change_21_23'].median().reset_index()
f=px.bar(b,x='quartile',y='days_cash_change_21_23',color_discrete_sequence=[RED])
f.update_xaxes(title='FY2021 days cash on hand, quartile')
f.update_yaxes(title='Median change in days cash, FY2021 to FY2023')
add(style(f,'7. The decline hit hospitals that had reserves','Same hospitals measured at both endpoints',height=440),
 'Hospitals starting with reserves lost 41 to 46 days. The bottom half did not move: the 25th percentile was 11.5 days in FY2021 and 10.7 in FY2023.')

Hospitals starting with reserves lost 41 to 46 days. The bottom half did not move: the 25th percentile was 11.5 days in FY2021 and 10.7 in FY2023.



---
# Part 4 &nbsp; Who joined which program

LIHNC membership and Epic pipeline participation, against geography and ownership.

In [12]:
# 8
g=h.dropna(subset=['rurality_band']).groupby(['rurality_band','program'],observed=True).size().reset_index(name='n')
g['share']=g.n/g.groupby('rurality_band',observed=True).n.transform('sum')
f=px.bar(g,x='rurality_band',y='share',color='program',color_discrete_map=PROGRAM,barmode='stack')
f.update_xaxes(title='Rurality of the hospital ZIP (RUCA band)'); f.update_yaxes(title='Share of hospitals',tickformat='.0%')
add(style(f,'8. The two programs reach opposite geographies',
 'RUCA 1-3 is metropolitan or metro-adjacent; 7-10 is small town and isolated'),
 'Epic participation rises with isolation (+0.24, p=0.013), LIHNC falls with it (-0.20, p=0.038), each controlling for the other. Only 1 of 12 hospitals in an urban ZIP is in the Epic pipeline.')

Epic participation rises with isolation (+0.24, p=0.013), LIHNC falls with it (-0.20, p=0.038), each controlling for the other. Only 1 of 12 hospitals in an urban ZIP is in the Epic pipeline.



In [13]:
# 9
g=h.groupby(['ownership','program']).size().reset_index(name='n')
g['share']=g.n/g.groupby('ownership').n.transform('sum')
f=px.bar(g,x='ownership',y='share',color='program',color_discrete_map=PROGRAM,barmode='stack')
f.update_xaxes(title='Ownership type'); f.update_yaxes(title='Share of hospitals',tickformat='.0%')
add(style(f,'9. LIHNC is a public hospital district network','Program participation by ownership type'),
 '21 of 24 LIHNC members are government districts. One of 13 proprietary hospitals is a member (Fisher p=0.003). The 12 proprietary hospitals outside the network are the unrecruited population, and their participation is decided by a corporate parent.')

21 of 24 LIHNC members are government districts. One of 13 proprietary hospitals is a member (Fisher p=0.003). The 12 proprietary hospitals outside the network are the unrecruited population, and their participation is decided by a corporate parent.



---
# Part 5 &nbsp; Medicaid and payment exposure

Enrollment decline at parish level, and dependence on the state payment stream that faces a phase-down from 2028.

In [14]:
# 10
x=h.dropna(subset=['medicaid_share_of_revenue','mco_pct_change_24_26'])
f=px.scatter(x,x='mco_pct_change_24_26',y='medicaid_share_of_revenue',color='program',
    size='medicaid_revenue',hover_name='hospital',color_discrete_map=PROGRAM,size_max=30)
f.update_xaxes(title='Parish Medicaid managed care enrollment change, Jan 2024 to May 2026',tickformat='.0%')
f.update_yaxes(title='Medicaid share of net patient revenue',tickformat='.0%')
xr=[x.mco_pct_change_24_26.min(),x.mco_pct_change_24_26.max()]
yr=[x.medicaid_share_of_revenue.min(),x.medicaid_share_of_revenue.max()]
quadrants(f,-0.25,x.medicaid_share_of_revenue.median(),
    ['Steeper loss,<br>less dependent','Milder loss,<br>less dependent',
     'Steeper loss, more dependent','Milder loss, more dependent'],xr,yr)
add(style(f,'10. Medicaid dependence against parish enrollment loss',
 'Dashed lines mark the statewide enrollment decline (-25%) and the median Medicaid revenue share'),
 'Medicaid is 23.9% of revenue at the median, far above its 1-3% inpatient day share. Statewide enrollment fell 25%, putting roughly $155M at risk. Two-thirds of that sits with hospitals in neither program.')

Medicaid is 23.9% of revenue at the median, far above its 1-3% inpatient day share. Statewide enrollment fell 25%, putting roughly $155M at risk. Two-thirds of that sits with hospitals in neither program.



In [15]:
# 11
p=h.groupby('parish').agg(hospitals=('hospital','size'),enrollment_change=('mco_pct_change_24_26','first'),
    state_payments=('ldh_payments_2025','sum')).reset_index().dropna(subset=['enrollment_change'])
p=p.nsmallest(20,'enrollment_change').sort_values('enrollment_change')
f=px.bar(p,x='enrollment_change',y='parish',orientation='h',color='hospitals',
    color_continuous_scale=['#c9d3e0',NAVY])
f.update_xaxes(title='Parish Medicaid enrollment change, Jan 2024 to May 2026',tickformat='.0%')
f.update_yaxes(title='Parish')
f.update_layout(coloraxis_colorbar=dict(title='Rural<br>hospitals'))
add(style(f,'11. Parishes losing Medicaid enrollment fastest','20 steepest declines, shaded by number of rural hospitals',
    height=680,left=150,bottom=90),
 'Cameron lost 38%, Plaquemines 34%, West Carroll 30%. West Carroll Memorial reports negative days cash and 34% Medicaid revenue, the worst pairing in the state.')

Cameron lost 38%, Plaquemines 34%, West Carroll 30%. West Carroll Memorial reports negative days cash and 34% Medicaid revenue, the worst pairing in the state.



---
# Part 6 &nbsp; What these hospitals actually do

Census, payer mix, pricing behavior and staffing. Several hospitals turn out to have almost no inpatient activity left.

In [16]:
# 12
o=h.dropna(subset=['adc_2024','beds']).copy()
o['dp_2025']=o.dp_2025.fillna(0)   # size cannot be NaN
f=px.scatter(o,x='beds',y='adc_2024',color='medicare_class',size='dp_2025',hover_name='hospital',
    color_discrete_sequence=[NAVY,TEAL,GOLD,RED,GREEN],size_max=30)
f.add_shape(type='line',x0=0,y0=0,x1=140,y1=140,line=dict(color=GREY,dash='dot',width=1))
f.add_hline(y=1,line=dict(color=RED,width=1,dash='dash'))
f.add_annotation(x=135,y=1,text='<b>1 patient per day</b>',showarrow=False,yanchor='bottom',xanchor='right',
    font=dict(size=10,color=RED),bgcolor='rgba(255,255,255,0.8)')
f.update_xaxes(title='Licensed beds'); f.update_yaxes(title='Average daily census (2024)',range=[-2,60])
add(style(f,'12. Ten hospitals have almost no inpatients',
 'Dotted line is full occupancy. Bubble size is SFY2025 directed payments'),
 'Ten hospitals average under one inpatient per day and all are CAHs receiving directed payments. West Ascension filed 4 inpatient days for the year, Lady of the Sea 11. These are already outpatient and emergency facilities holding acute licenses.')

Ten hospitals average under one inpatient per day and all are CAHs receiving directed payments. West Ascension filed 4 inpatient days for the year, Lady of the Sea 11. These are already outpatient and emergency facilities holding acute licenses.



In [17]:
# 13
x=h.dropna(subset=['specialty_facilities_zip','total_margin_fy2023','zip_population']).copy()
x=x[x.total_margin_fy2023.between(-.5,.5)]
f=px.scatter(x,x='zip_population',y='total_margin_fy2023',size='specialty_facilities_zip',
    color='specialty_facilities_zip',hover_name='hospital',color_continuous_scale=[GREEN,'#e8d9a8',RED],size_max=26)
f.add_hline(y=0,line=dict(color=GREY,width=1))
f.update_xaxes(title='Population of the hospital ZIP',type='log')
f.update_yaxes(title='Total margin (FY2023)',tickformat='.0%')
f.update_layout(coloraxis_colorbar=dict(title='Specialty<br>facilities<br>in ZIP'))
add(style(f,'13. Bigger towns, more competition, worse margins',
 'Specialty facilities are ambulatory surgery, imaging and specialty clinics in the same ZIP, not other hospitals',bottom=90),
 'The weaker financial cluster sits in ZIPs averaging 0.85 specialty facilities against 0.33 for the stronger one (p=0.005). Acute facility counts show no difference. Isolated hospitals hold a captive market.')

The weaker financial cluster sits in ZIPs averaging 0.85 specialty facilities against 0.33 for the stronger one (p=0.005). Acute facility counts show no difference. Isolated hospitals hold a captive market.



In [18]:
# 14
e=h.dropna(subset=['epic_cost_share_of_revenue','days_cash_on_hand_fy2023']).copy()
e['epic_est_pricing_m']=e.epic_est_pricing_m.fillna(0)
f=px.scatter(e,x='epic_cost_share_of_revenue',y='days_cash_on_hand_fy2023',color='epic_wave',
    size='epic_est_pricing_m',hover_name='hospital',text='hospital',
    color_discrete_map={'Wave 1':GREEN,'Wave 2':BLUE,'Wave 3+':GOLD},size_max=32)
f.update_traces(textposition='middle right',textfont=dict(size=8))
f.update_xaxes(title='Estimated Epic cost as a share of net patient revenue',tickformat='.0%')
f.update_yaxes(title='Days cash on hand (FY2023)',range=[-40,420])
quadrants(f,0.07,60,['Affordable, thin cash','Costly, thin cash','Affordable, funded','Costly but funded'],
          [0,0.29],[-40,420])
add(style(f,'14. Epic implementation burden against ability to fund it',
 'Dashed lines mark the median burden (7% of revenue) and 60 days cash. Bubble size is total estimated cost',height=600),
 'Per-seat pricing is flat near $61,000 regardless of hospital size, so burden ranges from 1% of revenue at Bunkie to 27% at Springhill. Opelousas is the concentration risk: 60% of Wave 1 cost, 49.7 days cash, negative on both margins.')

Per-seat pricing is flat near $61,000 regardless of hospital size, so burden ranges from 1% of revenue at Bunkie to 27% at Springhill. Opelousas is the concentration risk: 60% of Wave 1 cost, 49.7 days cash, negative on both margins.



In [19]:
# 15
ZCOL={'days_cash_on_hand_fy2023':'Days cash on hand','equity_ratio_fy2023':'Equity ratio',
 'patient_services_margin_fy2023':'Patient services margin','total_margin_fy2023':'Total margin',
 'non_care_margin_gap':'Non-care margin gap','state_payment_share_of_revenue':'State payment share',
 'medicaid_share_of_revenue':'Medicaid share','revenue_per_bed':'Revenue per bed',
 'revenue_per_fte':'Revenue per FTE','occupancy_rate':'Occupancy rate',
 'uncompensated_share_of_revenue':'Uncompensated care share','cost_to_charge':'Cost to charge',
 'ruca':'Rurality (RUCA)','zip_population':'ZIP population',
 'specialty_facilities_zip':'Specialty facilities','poverty_pct':'Poverty',
 'mco_pct_change_24_26':'Parish enrollment change','days_cash_change_21_23':'Cash trend 21-23'}
cm=h[list(ZCOL)].apply(pd.to_numeric,errors='coerce').corr(method='spearman').rename(index=ZCOL,columns=ZCOL)
f=px.imshow(cm.round(2),color_continuous_scale=[RED,'white',NAVY],zmin=-1,zmax=1,text_auto=True,aspect='auto')
f.update_traces(textfont=dict(size=8))
f.update_layout(coloraxis_colorbar=dict(title='Spearman<br>rho'))
f.update_xaxes(title='Metric'); f.update_yaxes(title='Metric')
add(style(f,'15. How the measures relate','Spearman correlation, 64 hospitals. Skewed distributions, so rank correlation rather than Pearson',
    height=720,left=170,bottom=170),
 'Three near-independent blocks: liquidity and capital structure, profitability, and place. Non-care margin gap tracks state payment share, which is chart 4. Rurality correlates with ZIP population and specialty facilities but not with margin directly, so geography acts on finances through competition.')

Three near-independent blocks: liquidity and capital structure, profitability, and place. Non-care margin gap tracks state payment share, which is chart 4. Rurality correlates with ZIP population and specialty facilities but not with margin directly, so geography acts on finances through competition.



In [20]:
# 16
s=h.dropna(subset=['days_cash_on_hand_fy2023_rank','patient_services_margin_fy2023']).copy()
s['net_patient_revenue_fy2023']=s.net_patient_revenue_fy2023.fillna(0)
f=px.scatter(s,x='patient_services_margin_fy2023',y='days_cash_on_hand_fy2023_rank',color='program',
    size='net_patient_revenue_fy2023',hover_name='hospital',color_discrete_map=PROGRAM,size_max=30)
f.update_xaxes(title='Patient services margin (FY2023)',tickformat='.0%',range=[-.75,.25])
f.update_yaxes(title='Liquidity rank within ownership type',tickformat='.0%')
quadrants(f,0,0.5,['Loses on care,<br>below-peer liquidity','Earns on care,<br>below-peer liquidity',
                   'Loses on care, above-peer liquidity','Earns on care, above-peer liquidity'],[-.75,.25],[0,1])
add(style(f,'16. Financial position: earnings against reserves',
 'Liquidity is a percentile rank within ownership type, since raw days cash measures treasury structure',height=580),
 'Program membership scatters across all four quadrants. Financial position does not predict LIHNC membership (accuracy 0.48 against a 0.60 base rate) or Epic participation (0.53 against 0.62).')

Program membership scatters across all four quadrants. Financial position does not predict LIHNC membership (accuracy 0.48 against a 0.60 base rate) or Epic participation (0.53 against 0.62).



---
# Part 7 &nbsp; Market position and shared infrastructure

Competition in the local market, Epic implementation burden, and the correlation structure across all measures.

In [21]:
# 17
m=h.dropna(subset=['medicaid_days_share','medicare_days_share']).copy()
m['other']=1-m.medicaid_days_share-m.medicare_days_share
g=m.groupby('medicare_class')[['medicare_days_share','medicaid_days_share','other']].median().reset_index()
g=g.melt(id_vars='medicare_class',var_name='payer',value_name='share')
g['payer']=g.payer.map({'medicare_days_share':'Medicare','medicaid_days_share':'Medicaid','other':'Commercial / other'})
f=px.bar(g,x='medicare_class',y='share',color='payer',barmode='stack',
    color_discrete_map={'Medicare':NAVY,'Medicaid':GOLD,'Commercial / other':'#c9d3e0'})
f.update_xaxes(title='Medicare reimbursement class'); f.update_yaxes(title='Share of inpatient days',tickformat='.0%')
add(style(f,'17. Inpatient payer mix by class','Median share of total inpatient days, FY2023'),
 'Medicaid is 1 to 3% of inpatient days everywhere except the rural referral centers at 27%, against 24% of net patient revenue. Medicaid volume in these hospitals is overwhelmingly outpatient and clinic, not inpatient.')

Medicaid is 1 to 3% of inpatient days everywhere except the rural referral centers at 27%, against 24% of net patient revenue. Medicaid volume in these hospitals is overwhelmingly outpatient and clinic, not inpatient.



In [22]:
# 18
u=h.dropna(subset=['uncompensated_share_of_revenue','medicaid_share_of_revenue']).copy()
u=u[u.uncompensated_share_of_revenue.between(0,.4)]
u['net_patient_revenue_fy2023']=u.net_patient_revenue_fy2023.fillna(0)
f=px.scatter(u,x='medicaid_share_of_revenue',y='uncompensated_share_of_revenue',color='ownership',
    size='net_patient_revenue_fy2023',hover_name='hospital',color_discrete_map=OWNER,size_max=28)
f.update_xaxes(title='Medicaid share of net patient revenue',tickformat='.0%')
f.update_yaxes(title='Uncompensated care cost as a share of net patient revenue',tickformat='.0%')
add(style(f,'18. Uncompensated care against Medicaid dependence','Both measured against FY2023 net patient revenue'),
 'This is how enrollment loss reaches a cost-settled hospital. Losing coverage does not cut the rate; it converts the patient to uninsured and moves the cost into this measure.')

This is how enrollment loss reaches a cost-settled hospital. Losing coverage does not cut the rate; it converts the patient to uninsured and moves the cost into this measure.



In [23]:
# 19
f=px.box(h.dropna(subset=['cost_to_charge']),x='medicare_class',y='cost_to_charge',
    color='medicare_class',color_discrete_sequence=[NAVY,TEAL,GOLD,RED,GREEN],points='all')
f.update_xaxes(title='Medicare reimbursement class')
f.update_yaxes(title='Cost-to-charge ratio')
add(style(f,'19. Pricing behavior inverts with size',
 'Cost-to-charge ratio: total cost over total billed charges. Higher means charges sit closer to actual cost',legend=False),
 'A CAH bills close to what care costs. A referral center bills roughly six times cost. Any shared-savings benchmark built on charges rather than cost will read these two groups very differently.')

A CAH bills close to what care costs. A referral center bills roughly six times cost. Any shared-savings benchmark built on charges rather than cost will read these two groups very differently.



In [24]:
# 20
s=h.dropna(subset=['fte_per_bed','revenue_per_fte']).copy()
s['beds']=s.beds.fillna(0)
f=px.scatter(s,x='fte_per_bed',y='revenue_per_fte',color='program',size='beds',
    hover_name='hospital',color_discrete_map=PROGRAM,size_max=28)
f.update_xaxes(title='FTE employees per licensed bed')
f.update_yaxes(title='Net patient revenue per FTE',tickprefix='$')
quadrants(f,s.fte_per_bed.median(),s.revenue_per_fte.median(),
 ['Lean staffing,<br>lower productivity','Heavy staffing,<br>lower productivity',
  'Lean staffing, higher productivity','Heavy staffing, higher productivity'],
 [0,s.fte_per_bed.max()],[0,s.revenue_per_fte.max()])
add(style(f,'20. Staffing intensity against revenue per employee','Dashed lines are the medians of each measure',height=560),
 'Heavy staffing with lower productivity is the signature of a hospital carrying standby capacity: staffed for a census it no longer runs. Several of the near-empty hospitals sit in that quadrant.')

Heavy staffing with lower productivity is the signature of a hospital carrying standby capacity: staffed for a census it no longer runs. Several of the near-empty hospitals sit in that quadrant.



In [25]:
# 21
v=h.dropna(subset=['social_vulnerability_index','diabetes_prevalence']).copy()
v['zip_population']=v.zip_population.fillna(0)
f=px.scatter(v,x='social_vulnerability_index',y='diabetes_prevalence',color='rurality_band',
    size='zip_population',hover_name='hospital',color_discrete_map=RURAL,size_max=28)
f.update_xaxes(title='Social vulnerability index (mean of poverty, food access, vehicle access, broadband)')
f.update_yaxes(title='Diabetes prevalence in the hospital ZIP')
add(style(f,'21. Community need around each hospital','ZIP-level measures from the Louisiana Health Atlas'),
 'None of these variables separates program members from non-members. Community need did not sort hospitals into LIHNC or the Epic pipeline, which is worth stating plainly if either is presented as needs-based.')

None of these variables separates program members from non-members. Community need did not sort hospitals into LIHNC or the Epic pipeline, which is worth stating plainly if either is presented as needs-based.



In [26]:
# 22
e=h.dropna(subset=['medicaid_class']).copy()
g=e.groupby(['medicaid_class','in_epic_pipeline']).size().reset_index(name='hospitals')
f=px.bar(g,x='hospitals',y='medicaid_class',color='in_epic_pipeline',orientation='h',barmode='stack',
    color_discrete_map={'Yes':GREEN,'No':GREY})
f.update_xaxes(title='Hospitals'); f.update_yaxes(title='Louisiana Medicaid class')
f.for_each_trace(lambda t: t.update(name='In Epic pipeline' if t.name=='Yes' else 'Not in pipeline'))
add(style(f,'22. Louisiana Medicaid class and Epic participation',
 'State classes set under the Rural Hospital Preservation Act, independent of the Medicare class',left=200,height=460),
 'Cost-settled classes dominate the universe. Epic participation spreads across them rather than concentrating in one, consistent with participation being unrelated to reimbursement structure.')

Cost-settled classes dominate the universe. Epic participation spreads across them rather than concentrating in one, consistent with participation being unrelated to reimbursement structure.



In [27]:
# 23
p=h[(h.ldh_payments_2024>0)|(h.ldh_payments_2025>0)].nlargest(14,'ldh_payments_2025').sort_values('ldh_payments_2025')
f=go.Figure()
f.add_trace(go.Bar(y=p.hospital,x=p.ldh_payments_2024,name='SFY2024',orientation='h',marker_color='#c9d3e0'))
f.add_trace(go.Bar(y=p.hospital,x=p.ldh_payments_2025,name='SFY2025',orientation='h',marker_color=NAVY))
f.update_xaxes(title='Total state Medicaid payments',tickprefix='$'); f.update_yaxes(title='Hospital')
add(style(f,'23. Largest recipients of state Medicaid payments',
 'Directed payments plus cost settlement, UPL and DSH, two state fiscal years',height=640,left=240),
 'These hospitals have the most to lose in the SFY2029 phase-down. Payments grew between the two years for most of them, setting a higher baseline for the 10-point annual reduction to work against.')
f.update_layout(barmode='group')

These hospitals have the most to lose in the SFY2029 phase-down. Payments grew between the two years for most of them, setting a higher baseline for the 10-point annual reduction to work against.



In [28]:
# 24
x=h.dropna(subset=['state_payment_share_of_revenue','days_cash_on_hand_fy2023']).copy()
x['ldh_payments_2025']=x.ldh_payments_2025.fillna(0)
x['cost_settled']=x.medicaid_class.astype(str).str.contains('Small Rural|Critical Access',case=False,na=False)
x['settlement']=np.where(x.cost_settled,'Cost settled','Not cost settled')
f=px.scatter(x,x='state_payment_share_of_revenue',y='days_cash_on_hand_fy2023',color='settlement',
    size='ldh_payments_2025',hover_name='hospital',
    color_discrete_map={'Cost settled':TEAL,'Not cost settled':RED},size_max=30)
f.update_xaxes(title='State Medicaid payments as a share of net patient revenue',tickformat='.0%')
f.update_yaxes(title='Days cash on hand (FY2023)',range=[-200,420])
quadrants(f,0.15,90,['Low dependence,<br>thin reserves','High dependence, thin reserves',
                     'Low dependence, funded','High dependence, funded'],
          [0,x.state_payment_share_of_revenue.max()],[-200,420])
add(style(f,'24. Exposure to the directed payment phase-down',
 'Dashed lines mark 15% revenue dependence and 90 days cash. Bubble size is total SFY2025 payments',height=580),
 'The lower right quadrant is high dependence with no reserves to absorb a reduction. Cost settlement offers no protection, because the exposure is the directed payment program rather than the settlement mechanism.')

The lower right quadrant is high dependence with no reserves to absorb a reduction. Cost settlement offers no protection, because the exposure is the directed payment program rather than the settlement mechanism.

